# Acompanhamento 2 - Pipeline e modelos preliminares

Este notebook documenta o segundo checkpoint do projeto de previsão de preços de imóveis.

- O foco é demonstrar o `pipeline.py` funcionando.
- Os modelos ainda são preliminares, mas já seguem um fluxo reprodutível.
- O treino parte do dataset bruto `data/treino.csv`.
- O `pipeline.py` recebe o CSV bruto de teste e executa limpeza, pré-processamento e predição.
- As métricas são comparadas com o baseline informado pelo professor.


## 1. Escopo do Acompanhamento 2

De acordo com o PDF do projeto, este checkpoint deve apresentar:

- estrutura do `pipeline.py` rodando;
- resultados preliminares dos primeiros modelos;
- fluxo de pre-processamento integrado ao modelo;
- capacidade de ler uma base nova e retornar predicoes sem erro.

O objetivo aqui nao e encerrar a competicao, mas mostrar que a base tecnica para a entrega final ja esta funcionando.

## 2. Organização dos datasets

Os arquivos ficam separados por etapa, mas a entrada oficial do Acompanhamento 2 é a base bruta.

- `data/treino.csv`: dataset bruto de treino, com `SalePrice` e valores ausentes originais.
- `data/teste_publico.csv`: dataset bruto de teste público, sem `SalePrice`, usado para validar o `pipeline.py`.
- `data/processed/treino_limpo.csv`: evidência do Acompanhamento 1; não é mais consumido diretamente no treino.
- `data/processed/teste_publico_limpo.csv`: evidência do Acompanhamento 1; não é mais consumido diretamente pelo `pipeline.py`.
- `acompanhamentos/acomp2/metricas_preliminares.csv`: resultados dos modelos preliminares.
- `acompanhamentos/acomp2/predicoes_teste_publico.csv`: predições geradas pelo pipeline a partir do teste público bruto.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "treino.csv").exists() and (candidate / "acompanhamentos" / "acomp2").exists():
            return candidate
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto.")


ROOT = find_project_root(Path.cwd().resolve())
ACOMP_DIR = ROOT / "acompanhamentos" / "acomp2"
DATA_DIR = ROOT / "data"
TRAIN_RAW_PATH = DATA_DIR / "treino.csv"
PUBLIC_TEST_RAW_PATH = DATA_DIR / "teste_publico.csv"
PROCESSED_DIR = DATA_DIR / "processed"
TRAIN_CLEAN_REFERENCE_PATH = PROCESSED_DIR / "treino_limpo.csv"
PUBLIC_TEST_CLEAN_REFERENCE_PATH = PROCESSED_DIR / "teste_publico_limpo.csv"
BASELINE_RMSLE = 0.17543

sys.path.insert(0, str(ACOMP_DIR))


## 3. Conferência dos dados brutos

Antes do treino, verificamos se os CSVs originais existem e mantêm o formato esperado.

- `treino.csv` deve conter a coluna alvo `SalePrice`.
- `teste_publico.csv` não deve conter `SalePrice`.
- As bases ainda possuem nulos, porque a limpeza agora ocorre dentro do `pipeline.py`.
- As estatísticas de limpeza são ajustadas somente no treino, evitando vazamento de dados.


In [2]:
for path in [TRAIN_RAW_PATH, PUBLIC_TEST_RAW_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Dataset bruto não encontrado: {path}")

train_df = pd.read_csv(TRAIN_RAW_PATH)
public_test_df = pd.read_csv(PUBLIC_TEST_RAW_PATH)

resumo_datasets = pd.DataFrame({
    "dataset": ["treino.csv", "teste_publico.csv"],
    "local": [str(TRAIN_RAW_PATH.relative_to(ROOT)), str(PUBLIC_TEST_RAW_PATH.relative_to(ROOT))],
    "linhas": [len(train_df), len(public_test_df)],
    "colunas": [train_df.shape[1], public_test_df.shape[1]],
    "possui_saleprice": ["SalePrice" in train_df.columns, "SalePrice" in public_test_df.columns],
    "nulos_total": [int(train_df.isna().sum().sum()), int(public_test_df.isna().sum().sum())],
})
resumo_datasets


,dataset,local,linhas,colunas,possui_saleprice,nulos_total
0,treino.csv,data/treino.csv,1168,81,True,6227
1,teste_publico.csv,data/teste_publico.csv,1459,80,False,7878


## 4. Estrutura do pipeline de modelagem

O script `treino_modelos.py` concentra o fluxo de treino e seleção do modelo.

- O treino lê `data/treino.csv`, ainda sujo.
- A primeira etapa do `sklearn.Pipeline` é `HousePricesCleaner`, definido em `pipeline.py`.
- `HousePricesCleaner` aplica a estratégia do Acompanhamento 1: ausência semântica como `Ausente`, zeros em áreas/contagens ausentes, `LotFrontage` por mediana do bairro e imputações residuais.
- Depois da limpeza, variáveis numéricas recebem imputação por mediana e escalonamento.
- Variáveis categóricas recebem imputação com `Ausente` e One-Hot Encoding.
- O alvo `SalePrice` é transformado com `log1p` durante o treino e volta para dólares com `expm1`.
- O melhor modelo é salvo em `modelo_acomp2.joblib`, já com limpeza, pré-processamento e modelo.


## 5. Modelos preliminares testados

Foram escolhidos modelos iniciais com complexidade crescente.

- `Ridge`: baseline linear regularizado.
- `RandomForest`: modelo de ensemble baseado em arvores.
- `GradientBoosting`: ensemble sequencial, geralmente forte para dados tabulares.

As metricas avaliadas sao:

- `RMSLE`: metrica principal da competicao.
- `RMSE`: erro quadratico medio em dolares.
- `MAE`: erro absoluto medio em dolares.
- `R2`: proporcao da variancia explicada.
- `tempo_treino_s`: tempo de treino em segundos.

In [3]:
from treino_modelos import main as treinar_modelos

metricas = treinar_modelos()
metricas

Metricas preliminares:
          modelo   rmsle        rmse         mae      r2  tempo_treino_s
GradientBoosting 0.13089 24940.34612 15485.41469 0.90377         2.83078
           Ridge 0.13639 26496.65928 16338.38989 0.89138         0.10645
    RandomForest 0.15670 34518.28347 19711.87369 0.81566         0.72421

Modelo selecionado: GradientBoosting
Modelo salvo em: /home/arthur/dev/unifor/ciência de dados/acompanhamentos/acomp2/modelo_acomp2.joblib
Predicoes publicas salvas em: /home/arthur/dev/unifor/ciência de dados/acompanhamentos/acomp2/predicoes_teste_publico.csv


,modelo,rmsle,rmse,mae,r2,tempo_treino_s
0,GradientBoosting,0.130892,24940.346120,15485.414689,0.903769,2.830781
1,Ridge,0.136390,26496.659281,16338.389888,0.891384,0.106446
2,RandomForest,0.156703,34518.283472,19711.873691,0.815664,0.724208


## 6. Comparacao com o baseline do professor

O arquivo `docs/metricas_baseline.txt` informa RMSLE de `0.17543` para a regressao linear simples do professor.

- Valores menores de RMSLE sao melhores.
- Um modelo supera o baseline quando `rmsle < 0.17543`.
- Essa comparacao e preliminar, pois aqui usamos validacao local.

In [4]:
comparacao_baseline = metricas.copy()
comparacao_baseline["baseline_professor_rmsle"] = BASELINE_RMSLE
comparacao_baseline["supera_baseline"] = comparacao_baseline["rmsle"] < BASELINE_RMSLE
comparacao_baseline

,modelo,rmsle,rmse,mae,r2,tempo_treino_s,baseline_professor_rmsle,supera_baseline
0,GradientBoosting,0.130892,24940.346120,15485.414689,0.903769,2.830781,0.17543,True
1,Ridge,0.136390,26496.659281,16338.389888,0.891384,0.106446,0.17543,True
2,RandomForest,0.156703,34518.283472,19711.873691,0.815664,0.724208,0.17543,True


## 7. Validação do `pipeline.py`

O `pipeline.py` deve estar pronto para receber um CSV bruto e retornar apenas as predições.

Nesta validação, conferimos que:

- a função `predict()` aceita `data/teste_publico.csv`, ainda com nulos;
- a saída é um `np.ndarray`;
- a quantidade de predições é igual à quantidade de linhas do dataset;
- todas as predições são finitas;
- o retorno não inclui coluna de ID nem coluna de texto.


In [5]:
from pipeline import predict

predicoes = predict(PUBLIC_TEST_RAW_PATH)
validacao_pipeline = {
    "entrada": str(PUBLIC_TEST_RAW_PATH.relative_to(ROOT)),
    "tipo_saida": type(predicoes).__name__,
    "linhas_preditas": len(predicoes),
    "linhas_entrada": len(public_test_df),
    "somente_valores_finitos": bool(np.isfinite(predicoes).all()),
    "min_predicao": float(predicoes.min()),
    "max_predicao": float(predicoes.max()),
}
validacao_pipeline


{'entrada': 'data/teste_publico.csv',
 'tipo_saida': 'ndarray',
 'linhas_preditas': 1459,
 'linhas_entrada': 1459,
 'somente_valores_finitos': True,
 'min_predicao': 45700.81103620755,
 'max_predicao': 514390.8829870194}

## 8. Amostra das predições geradas

As predições do teste público ficam salvas dentro da pasta do acompanhamento.

- O arquivo é apenas uma evidência para apresentação.
- O retorno exigido pelo avaliador continua sendo o array produzido por `pipeline.py`.
- A ordem das linhas segue a ordem de `data/teste_publico.csv`.


In [6]:
pd.read_csv(ACOMP_DIR / "predicoes_teste_publico.csv").head()

,Id,SalePrice_predito
0,1461,127033.891617
1,1462,157756.492378
2,1463,176509.452121
3,1464,188250.652703
4,1465,194304.241613


## 9. Conclusão preliminar

- O fluxo de modelagem parte dos datasets brutos em `data/`.
- A limpeza saiu do script de treino como etapa manual e foi encapsulada em `pipeline.py`.
- O modelo salvo em `modelo_acomp2.joblib` contém limpeza, pré-processamento e regressor.
- O `pipeline.py` foi validado com a base bruta de teste público.
- Essa estrutura está alinhada ao PDF, pois o avaliador pode enviar um CSV novo e o script processa tudo de forma autônoma.
